In [1]:
# SIMULATED A/A: allocation check on historical user identifiers.
# No actual online experiment or treatment effect.

import math

n_a = 126_561
n_b = 127_048

total_users = n_a + n_b
expected_per_group = total_users / 2

# Pearson chi-square statistic for the planned 50:50 allocation.
chi2_stat = (
    (n_a - expected_per_group) ** 2 / expected_per_group
    + (n_b - expected_per_group) ** 2 / expected_per_group
)

# For 1 degree of freedom, this equals the chi-square upper-tail probability.
p_value = math.erfc(math.sqrt(chi2_stat / 2))

# Fixed diagnostic threshold; do not change it to obtain a desired result.
srm_alpha = 0.001

print(f"Total users: {total_users:,}")
print(f"Expected users per group: {expected_per_group:,.1f}")
print(f"A allocation: {n_a / total_users:.4%}")
print(f"B allocation: {n_b / total_users:.4%}")
print(f"Chi-square statistic: {chi2_stat:.6f}")
print(f"SRM p-value: {p_value:.6f}")
print(f"SRM threshold: {srm_alpha}")
print(f"SRM flag: {p_value < srm_alpha}")

Total users: 253,609
Expected users per group: 126,804.5
A allocation: 49.9040%
B allocation: 50.0960%
Chi-square statistic: 0.935176
SRM p-value: 0.333522
SRM threshold: 0.001
SRM flag: False


In [2]:
# SIMULATED A/A ONLY — no product treatment or causal effect.
import math
from statistics import NormalDist

n_a, n_b = 126_561, 127_048
buyers_a, buyers_b = 1_652, 1_691

rate_a = buyers_a / n_a
rate_b = buyers_b / n_b

# Difference is always B minus A.
difference = rate_b - rate_a
relative_difference = difference / rate_a

# Two-sided two-proportion z-test.
# Under the null hypothesis, both groups share a common purchase rate.
pooled_rate = (buyers_a + buyers_b) / (n_a + n_b)
se_null = math.sqrt(
    pooled_rate * (1 - pooled_rate) * (1 / n_a + 1 / n_b)
)
z_stat = difference / se_null
p_value = math.erfc(abs(z_stat) / math.sqrt(2))

# Large-sample 95% confidence interval for the absolute difference.
# Unlike the test, this uses each group's observed rate.
se_difference = math.sqrt(
    rate_a * (1 - rate_a) / n_a
    + rate_b * (1 - rate_b) / n_b
)

alpha = 0.05
z_critical = NormalDist().inv_cdf(1 - alpha / 2)
ci_lower = difference - z_critical * se_difference
ci_upper = difference + z_critical * se_difference

print(f"A purchase rate: {rate_a:.4%}")
print(f"B purchase rate: {rate_b:.4%}")
print(f"Difference B - A: {difference * 100:+.4f} percentage points")
print(f"Relative difference: {relative_difference:+.2%}")
print(f"Z statistic: {z_stat:.6f}")
print(f"Two-sided p-value: {p_value:.6f}")
print(
    "95% CI for B - A: "
    f"[{ci_lower * 100:+.4f}, {ci_upper * 100:+.4f}] percentage points"
)
print(f"Significant at 5%: {p_value < alpha}")

A purchase rate: 1.3053%
B purchase rate: 1.3310%
Difference B - A: +0.0257 percentage points
Relative difference: +1.97%
Z statistic: 0.567247
Two-sided p-value: 0.570547
95% CI for B - A: [-0.0631, +0.1145] percentage points
Significant at 5%: False


In [3]:
# HYPOTHETICAL EXPERIMENT PLANNING.
# These assumed improvements are not observed treatment effects.

import math
from statistics import NormalDist

baseline = 3_343 / 253_609
alpha = 0.05
power = 0.80

normal = NormalDist()
z_alpha = normal.inv_cdf(1 - alpha / 2)
z_power = normal.inv_cdf(power)

print(f"Historical baseline: {baseline:.4%}")
print(f"Two-sided alpha: {alpha:.0%}; target power: {power:.0%}")
print()
print(
    f"{'Relative lift':>14} "
    f"{'Target rate':>14} "
    f"{'Absolute MDE (pp)':>19} "
    f"{'Users/group':>14} "
    f"{'Total users':>14}"
)

for relative_lift in [0.05, 0.10, 0.20]:
    target_rate = baseline * (1 + relative_lift)
    absolute_mde = target_rate - baseline
    average_rate = (baseline + target_rate) / 2

    # Normal approximation for two independent proportions,
    # equal group sizes, without continuity correction.
    numerator = (
        z_alpha * math.sqrt(2 * average_rate * (1 - average_rate))
        + z_power * math.sqrt(
            baseline * (1 - baseline)
            + target_rate * (1 - target_rate)
        )
    ) ** 2

    users_per_group = math.ceil(numerator / absolute_mde**2)

    print(
        f"{relative_lift:>13.0%} "
        f"{target_rate:>13.4%} "
        f"{absolute_mde * 100:>19.4f} "
        f"{users_per_group:>14,} "
        f"{2 * users_per_group:>14,}"
    )

Historical baseline: 1.3182%
Two-sided alpha: 5%; target power: 80%

 Relative lift    Target rate   Absolute MDE (pp)    Users/group    Total users
           5%       1.3841%              0.0659        481,661        963,322
          10%       1.4500%              0.1318        123,310        246,620
          20%       1.5818%              0.2636         32,274         64,548


In [4]:
# HYPOTHETICAL DESIGN SENSITIVITY, not observed/post-hoc power.
import math
from statistics import NormalDist

baseline = 3_343 / 253_609
alpha = 0.05
target_power = 0.80

# Use the smaller group for a conservative equal-size approximation.
available_per_group = min(126_561, 127_048)

normal = NormalDist()
z_alpha = normal.inv_cdf(1 - alpha / 2)
z_power = normal.inv_cdf(target_power)

def required_users_per_group(relative_lift):
    target_rate = baseline * (1 + relative_lift)
    difference = target_rate - baseline
    average_rate = (baseline + target_rate) / 2

    numerator = (
        z_alpha * math.sqrt(2 * average_rate * (1 - average_rate))
        + z_power * math.sqrt(
            baseline * (1 - baseline)
            + target_rate * (1 - target_rate)
        )
    ) ** 2

    return numerator / difference**2

# Search between a 0.01% and a 100% relative improvement.
lower = 0.0001
upper = 1.0

assert required_users_per_group(lower) > available_per_group
assert required_users_per_group(upper) <= available_per_group

for _ in range(80):
    midpoint = (lower + upper) / 2

    if required_users_per_group(midpoint) > available_per_group:
        lower = midpoint
    else:
        upper = midpoint

relative_mde = upper
absolute_mde = baseline * relative_mde
detectable_target_rate = baseline + absolute_mde

print(f"Planning users per group: {available_per_group:,}")
print(f"Baseline purchase rate: {baseline:.4%}")
print(f"Approximate relative MDE: {relative_mde:.2%}")
print(f"Approximate absolute MDE: {absolute_mde * 100:.4f} percentage points")
print(f"Corresponding target rate: {detectable_target_rate:.4%}")
print(f"Target power: {target_power:.0%}")

Planning users per group: 126,561
Baseline purchase rate: 1.3182%
Approximate relative MDE: 9.87%
Approximate absolute MDE: 0.1301 percentage points
Corresponding target rate: 1.4482%
Target power: 80%


In [5]:
# SIMULATED A/A: first-observed device balance.
import math

n_a, n_b = 126_561, 127_048

device_counts = {
    "desktop": (73_624, 73_666),
    "mobile": (50_101, 50_523),
    "tablet": (2_836, 2_859),
}

assert sum(a for a, b in device_counts.values()) == n_a
assert sum(b for a, b in device_counts.values()) == n_b

print(
    f"{'Device':<10} {'A share':>10} {'B share':>10} "
    f"{'B-A (pp)':>12} {'SMD':>10}"
)

absolute_smds = []

for device, (count_a, count_b) in device_counts.items():
    p_a = count_a / n_a
    p_b = count_b / n_b

    # Treat each category as a binary indicator:
    # e.g. desktop = 1, other devices = 0.
    pooled_sd = math.sqrt(
        (p_a * (1 - p_a) + p_b * (1 - p_b)) / 2
    )
    smd = (p_b - p_a) / pooled_sd
    absolute_smds.append(abs(smd))

    print(
        f"{device:<10} {p_a:>9.4%} {p_b:>9.4%} "
        f"{(p_b - p_a) * 100:>+12.4f} {smd:>+10.6f}"
    )

print(f"\nMaximum absolute SMD: {max(absolute_smds):.6f}")

Device        A share    B share     B-A (pp)        SMD
desktop     58.1727%  57.9828%      -0.1899  -0.003849
mobile      39.5864%  39.7669%      +0.1804  +0.003688
tablet       2.2408%   2.2503%      +0.0095  +0.000642

Maximum absolute SMD: 0.003849
